# Week 3 Reranking on Kaggle

In [ ]:
# Fill these before running.

GITHUB_REPO_URL = "https://github.com/Jasmine-Zhuang/faithfulness-guided-reranking.git"
GIT_REF = "main"  # branch, tag, or commit

# Set to None if you do not want to run that dataset.
XSUM_INPUT_JSONL = "/kaggle/input/fgr-reranking-inputs/xsum/validation_k5_candidates.jsonl"
CNN_INPUT_JSONL = "/kaggle/input/fgr-reranking-inputs/cnn_dailymail/validation_k5_candidates.jsonl"

# Use "cuda" on Kaggle GPU notebooks. Change to "cpu" if needed.
DEVICE = "cuda"

# Optional smoke test size. Set to None for full run.
NUM_EXAMPLES = None

In [ ]:
from pathlib import Path

if XSUM_INPUT_JSONL is not None:
    assert Path(XSUM_INPUT_JSONL).exists(), f"Missing XSum JSONL: {XSUM_INPUT_JSONL}"
if CNN_INPUT_JSONL is not None:
    assert Path(CNN_INPUT_JSONL).exists(), f"Missing CNN JSONL: {CNN_INPUT_JSONL}"

print("Repo:", GITHUB_REPO_URL)
print("Ref:", GIT_REF)
print("XSum:", XSUM_INPUT_JSONL)
print("CNN:", CNN_INPUT_JSONL)
print("Device:", DEVICE)
print("Num examples:", NUM_EXAMPLES)

In [ ]:
!rm -rf /kaggle/working/faithfulness-guided-reranking
!git clone {GITHUB_REPO_URL} /kaggle/working/faithfulness-guided-reranking
!cd /kaggle/working/faithfulness-guided-reranking && git checkout {GIT_REF}
!mkdir -p /kaggle/working/faithfulness-guided-reranking/outputs/xsum
!mkdir -p /kaggle/working/faithfulness-guided-reranking/outputs/cnn_dailymail

In [ ]:
!python -m pip install --upgrade pip
!pip install -r /kaggle/working/faithfulness-guided-reranking/requirements.txt
!pip install summac==0.0.4 nltk sentencepiece protobuf

In [ ]:
from pathlib import Path
import shutil

workdir = Path('/kaggle/working/faithfulness-guided-reranking')

if XSUM_INPUT_JSONL is not None:
    dst = workdir / 'outputs' / 'xsum' / 'validation_k5_candidates.jsonl'
    shutil.copy2(XSUM_INPUT_JSONL, dst)
    print('Copied', dst)

if CNN_INPUT_JSONL is not None:
    dst = workdir / 'outputs' / 'cnn_dailymail' / 'validation_k5_candidates.jsonl'
    shutil.copy2(CNN_INPUT_JSONL, dst)
    print('Copied', dst)

## Run XSum

In [ ]:
if XSUM_INPUT_JSONL is None:
    print('Skip XSum')
else:
    extra = '' if NUM_EXAMPLES is None else f' --num-examples {NUM_EXAMPLES}'
    cmd = (
        'cd /kaggle/working/faithfulness-guided-reranking && '
        'PYTHONPATH=src python scripts/run_week3_reranking.py '
        '--input outputs/xsum/validation_k5_candidates.jsonl '
        f'--device {DEVICE}' + extra
    )
    print(cmd)
    !$cmd

## Run CNN/DailyMail

In [ ]:
if CNN_INPUT_JSONL is None:
    print('Skip CNN/DailyMail')
else:
    extra = '' if NUM_EXAMPLES is None else f' --num-examples {NUM_EXAMPLES}'
    cmd = (
        'cd /kaggle/working/faithfulness-guided-reranking && '
        'PYTHONPATH=src python scripts/run_week3_reranking.py '
        '--input outputs/cnn_dailymail/validation_k5_candidates.jsonl '
        f'--device {DEVICE}' + extra
    )
    print(cmd)
    !$cmd

## Output Paths

In [ ]:
!find /kaggle/working/faithfulness-guided-reranking/outputs -maxdepth 3 -type f | sort